# Flood Damage Assessment App

Self-contained Gradio app for comparing pre-flood and post-flood images. This notebook includes demo imagery, image validation, change and water detection, severity estimation, weather fusion, population estimates, and the interactive interface.

Run the setup cell once, then run the app code cell and the final launch cell. The app opens locally at http://127.0.0.1:7860/.

In [1]:
%pip install "gradio>=5,<7" "numpy>=1.26" "opencv-python>=4.9"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import numpy as np
import gradio as gr


def create_flood_demo():
    height, width = 512, 768
    pre = np.zeros((height, width, 3), dtype=np.uint8)
    pre[:] = [120, 140, 100]
    cv2.rectangle(pre, (0, 0), (768, 120), (140, 150, 110), -1)
    cv2.rectangle(pre, (0, 390), (768, 512), (130, 145, 105), -1)
    cv2.line(pre, (0, 350), (768, 120), (170, 160, 135), 20)
    cv2.line(pre, (150, 0), (300, 512), (170, 160, 135), 15)

    buildings = [
        (320, 50, 390, 110), (430, 60, 510, 125), (550, 40, 640, 110),
        (280, 180, 365, 250), (400, 185, 480, 255), (520, 165, 610, 240),
        (330, 310, 420, 370), (470, 300, 560, 365), (590, 285, 675, 350),
    ]
    for x1, y1, x2, y2 in buildings:
        cv2.rectangle(pre, (x1, y1), (x2, y2), (190, 185, 170), -1)
        cv2.rectangle(pre, (x1 + 8, y1 + 8), (x2 - 8, y2 - 8), (210, 205, 190), -1)

    post = pre.copy()
    flood_mask = np.zeros((height, width), dtype=np.uint8)
    cv2.ellipse(flood_mask, (230, 330), (220, 130), 0, 0, 360, 255, -1)
    cv2.rectangle(flood_mask, (0, 390), (768, 512), 255, -1)
    post[flood_mask > 0] = [70, 105, 145]
    points = np.array([[120, 250], [270, 160], [380, 190], [340, 300], [200, 340]])
    cv2.fillPoly(flood_mask, [points], 255)
    post[flood_mask > 0] = [65, 100, 140]

    damaged_buildings = [(280, 180, 365, 250), (520, 165, 610, 240), (470, 300, 560, 365)]
    for x1, y1, x2, y2 in damaged_buildings:
        cv2.rectangle(post, (x1, y1), (x2, y2), (80, 75, 70), -1)
    return pre, post, flood_mask


def prepare_image(image):
    if image is None:
        raise gr.Error("Upload both the before-flood and after-flood images.")
    image = np.asarray(image)
    if image.size == 0:
        raise gr.Error("The uploaded image is empty. Choose a valid image file.")
    if image.ndim == 3 and image.shape[-1] == 1:
        image = image[:, :, 0]
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    elif image.ndim == 3 and image.shape[-1] == 4:
        image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
    elif image.ndim != 3 or image.shape[-1] != 3:
        raise gr.Error("Images must be grayscale, RGB, or RGBA.")
    if not np.issubdtype(image.dtype, np.number) or not np.isfinite(image).all():
        raise gr.Error("The uploaded image contains invalid pixel values.")
    if np.issubdtype(image.dtype, np.floating) and image.size and image.max() <= 1.0 and image.min() >= 0.0:
        image = image * 255.0
    image = np.clip(image, 0, 255).astype(np.uint8)
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR)


def flood_damage_pipeline(pre_image, post_image, rainfall_72h=120, peak_intensity=18, humidity=82, population_density=1200):
    pre = prepare_image(pre_image)
    post = prepare_image(post_image)
    if pre.shape[:2] != post.shape[:2]:
        interpolation = cv2.INTER_AREA if post.shape[0] > pre.shape[0] or post.shape[1] > pre.shape[1] else cv2.INTER_LINEAR
        post = cv2.resize(post, (pre.shape[1], pre.shape[0]), interpolation=interpolation)

    gray_pre = cv2.cvtColor(pre, cv2.COLOR_BGR2GRAY)
    gray_post = cv2.cvtColor(post, cv2.COLOR_BGR2GRAY)
    difference = cv2.GaussianBlur(cv2.absdiff(gray_pre, gray_post), (5, 5), 0)
    threshold = np.percentile(difference, 88)
    change_mask = (difference > threshold).astype(np.uint8) * 255
    kernel = np.ones((5, 5), np.uint8)
    change_mask = cv2.morphologyEx(change_mask, cv2.MORPH_OPEN, kernel)
    change_mask = cv2.morphologyEx(change_mask, cv2.MORPH_CLOSE, kernel)
    changed = change_mask > 0

    red = post[:, :, 0].astype(np.int16)
    green = post[:, :, 1].astype(np.int16)
    blue = post[:, :, 2].astype(np.int16)
    water_mask = ((blue - red > 15) & (blue > green)) & changed

    rain_score = min(rainfall_72h / 150, 1.0)
    peak_score = min(peak_intensity / 30, 1.0)
    humidity_score = humidity / 100.0
    weather_score = 0.35 * rain_score + 0.35 * peak_score + 0.30 * humidity_score
    change_score = difference.astype(np.float32) / 255.0
    severity_score = np.clip(0.60 * change_score + 0.25 * water_mask.astype(float) + 0.15 * weather_score, 0, 1)
    severe = (severity_score >= 0.65) & changed
    moderate = (severity_score >= 0.40) & (severity_score < 0.65) & changed
    peripheral = (severity_score < 0.40) & changed

    visual_strength = np.mean(difference[changed]) / 255.0 if np.any(changed) else 0.0
    confidence = float(np.clip(0.70 * visual_strength + 0.30 * weather_score, 0, 1))
    height, width = post.shape[:2]
    affected = severe | moderate | peripheral
    affected_area = float(np.sum(affected) / (height * width) * 4.0)
    affected_population = affected_area * population_density
    review = "MANUAL REVIEW REQUIRED" if confidence < 0.60 else "HIGH CONFIDENCE"

    overlay = post.copy()
    overlay[peripheral] = [255, 200, 0]
    overlay[moderate] = [255, 100, 0]
    overlay[severe] = [220, 0, 0]
    damage_map = cv2.addWeighted(post, 0.55, overlay, 0.45, 0)
    total_pixels = height * width
    results = {
        "Flood/Wet Area (%)": round(float(np.sum(water_mask) / total_pixels * 100), 2),
        "Severe Damage (%)": round(float(np.sum(severe) / total_pixels * 100), 2),
        "Moderate Damage (%)": round(float(np.sum(moderate) / total_pixels * 100), 2),
        "Peripheral Risk (%)": round(float(np.sum(peripheral) / total_pixels * 100), 2),
        "Affected Area (km²)": round(affected_area, 2),
        "Estimated Affected Population": round(float(affected_population)),
        "Confidence (%)": round(confidence * 100, 1),
        "Weather Corroboration (%)": round(float(weather_score * 100), 1),
        "Review Status": review,
    }
    summary = (
        "### Flood Damage Assessment Summary\n"
        f"- Flood/Wet Area: {results['Flood/Wet Area (%)']:.2f}%\n"
        f"- Severe Damage: {results['Severe Damage (%)']:.2f}%\n"
        f"- Moderate Damage: {results['Moderate Damage (%)']:.2f}%\n"
        f"- Peripheral Risk: {results['Peripheral Risk (%)']:.2f}%\n"
        f"- Affected Area: {results['Affected Area (km²)']:.2f} km²\n"
        f"- Estimated Population: {results['Estimated Affected Population']}\n"
        f"- Confidence: {results['Confidence (%)']:.1f}%\n"
        f"- Status: {review}"
    )
    return (
        cv2.cvtColor(damage_map, cv2.COLOR_BGR2RGB),
        cv2.cvtColor(change_mask, cv2.COLOR_GRAY2RGB),
        water_mask.astype(np.uint8) * 255,
        results,
        summary,
    )


def analyze_images(pre_image, post_image, rainfall, peak, humidity, population_density):
    return flood_damage_pipeline(pre_image, post_image, rainfall, peak, humidity, population_density)


pre_demo, post_demo, _ = create_flood_demo()
pre_demo = cv2.cvtColor(pre_demo, cv2.COLOR_BGR2RGB)
post_demo = cv2.cvtColor(post_demo, cv2.COLOR_BGR2RGB)

with gr.Blocks(title="Flood Damage Assessment") as demo:
    gr.Markdown("# Flood Damage Assessment")
    gr.Markdown("Upload before and after flood images, or use the preloaded demo scene.")
    with gr.Row():
        pre_input = gr.Image(label="Before Flood (Pre-image)", type="numpy", value=pre_demo)
        post_input = gr.Image(label="After Flood (Post-image)", type="numpy", value=post_demo)
    with gr.Row():
        rainfall = gr.Slider(0, 300, value=120, step=1, label="72h Rainfall (mm)")
        peak = gr.Slider(0, 60, value=18, step=1, label="Peak Intensity (mm/h)")
        humidity = gr.Slider(0, 100, value=82, step=1, label="Humidity (%)")
        population = gr.Slider(0, 5000, value=1200, step=10, label="Population Density (people/km²)")
    analyze = gr.Button("Analyze Flood Damage", variant="primary")
    with gr.Row():
        damage_output = gr.Image(label="Damage Severity Map")
        change_output = gr.Image(label="Change Detection Mask")
        water_output = gr.Image(label="Water Detection Mask")
    result_json = gr.JSON(label="Assessment Summary")
    result_text = gr.Markdown()
    analyze.click(
        fn=analyze_images,
        inputs=[pre_input, post_input, rainfall, peak, humidity, population],
        outputs=[damage_output, change_output, water_output, result_json, result_text],
    )

In [ ]:
# Confirm the pipeline accepts different resolutions and aspect ratios.
from_demo_pre, from_demo_post, _ = create_flood_demo()
pre_rgb_test = cv2.cvtColor(from_demo_pre, cv2.COLOR_BGR2RGB)
post_rgb_test = cv2.cvtColor(from_demo_post, cv2.COLOR_BGR2RGB)

portrait_post = cv2.resize(post_rgb_test, (320, 512))
portrait_outputs = analyze_images(pre_rgb_test, portrait_post, 120, 18, 82, 1200)
assert portrait_outputs[0].shape == pre_rgb_test.shape

small_post = cv2.resize(post_rgb_test, (384, 256))
small_outputs = analyze_images(pre_rgb_test, small_post, 120, 18, 82, 1200)
assert small_outputs[0].shape == pre_rgb_test.shape

print("Mixed aspect-ratio and resolution checks passed.")

## Use

1. Run the dependency cell once in the notebook kernel.
2. Run the app code cell to define the pipeline and Gradio interface.
3. Run the final launch cell. Open the local Gradio URL printed by the cell.
4. Demo images are preloaded. Upload your own before/after pair to replace them. Interrupt the launch cell to stop the server.

This is a demo assessment pipeline, not a substitute for validated remote-sensing analysis or field verification. Do not commit real incident imagery or sensitive data with this notebook.

In [5]:
demo.launch(server_name="127.0.0.1", share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
